In [2]:
# inlegalbert_bilstm_mha_crf_spl_v4.py  (SELF-PACED LEARNING REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# NEW in v4 (Self-Paced Learning for Class Imbalance):
#   1.  SelfPacedLearner class  — manages pacing schedule & sample selection
#   2.  Per-document loss tracking  — losses recomputed every SPL_UPDATE_FREQ epochs
#   3.  Pacing functions  — linear | exponential | logarithmic | root
#   4.  Rare-class document boost  — minority-class docs get lowered effective loss
#       so they are included EARLIER in training, countering imbalance
#   5.  SPL_MIN_SAMPLES floor  — model always sees at least this fraction of data
#   6.  Class-weighted SPL loss  — inside selected batch, rare-class tokens are
#       up-weighted by inverse-frequency weights (SPL handles WHICH docs to train
#       on; class weights handle HOW MUCH to penalise each token inside the batch)
#   7.  SPL diagnostics logged per epoch  — ratio selected, mean/std loss, rare
#       class doc inclusion rate saved to history CSV
#
# Everything from v3 is preserved (SWA, cosine LR, layer-wise decay, etc.)
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG  (v3 settings preserved)
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_spl_v4_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 80
BERT_LR         = 1e-5
HEAD_LR         = 2e-4
WEIGHT_DECAY    = 0.1
GRAD_CLIP       = 1.0
DROPOUT         = 0.5

BERT_FREEZE_LAYERS  = 10
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1

MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1

AUX_CE_WEIGHT    = 0.3
LABEL_SMOOTHING  = 0.1

SWA_START_FRAC   = 0.80
SWA_LR           = 5e-5

ES_PATIENCE      = 8
ES_MIN_DELTA     = 1e-4

GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_RATIO    = 0.05
RARE_THRESHOLD  = 0.05

# ── Self-Paced Learning (NEW) ──────────────────────────────
SPL_ENABLED        = True
# What fraction of training docs to start with (easiest first)
SPL_INITIAL_RATIO  = 0.40
# Grow to using all training docs by the end
SPL_FINAL_RATIO    = 1.00
# How the curriculum grows: "linear" | "exponential" | "logarithmic" | "root"
#   linear      → uniform growth
#   exponential → slow growth early, rapid late  (default, standard SPL)
#   logarithmic → rapid growth early, slow late  (aggressive warm-up)
#   root        → fast then slow (compromise)
SPL_PACING         = "exponential"
# Rare-class docs are treated as having their loss divided by this factor,
# so they look "easier" → included earlier to fight class imbalance
SPL_RARE_BOOST     = 2.0
# Never drop below this fraction of the training set (safety floor)
SPL_MIN_RATIO      = 0.15
# How often to recompute per-doc losses (every N epochs)
# Set to 1 for maximum accuracy; higher = faster training
SPL_UPDATE_FREQ    = 2
# Use inverse-frequency class weights inside selected batch (extra imbalance fix)
SPL_USE_CLASS_WEIGHTS = True

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# SELF-PACED LEARNER  (NEW)
# ═══════════════════════════════════════════════════════════
class SelfPacedLearner:
    """
    Implements Self-Paced Learning with class-imbalance awareness.

    Core idea
    ---------
    - At each epoch, rank all training documents by their loss.
    - Select only the easiest `k` documents (lowest loss), where `k`
      grows from SPL_INITIAL_RATIO → SPL_FINAL_RATIO over training.
    - Rare-class documents are boosted: their effective loss is divided
      by SPL_RARE_BOOST, making them appear easier → included earlier,
      counteracting the model's tendency to ignore minority classes.

    Pacing functions (how `k` grows with epoch)
    -------------------------------------------
    linear      : uniform pace
    exponential : slow at first, then rapid — model cements easy
                  patterns before tackling rare/hard ones
    logarithmic : fast at first (aggressive warm-up), then slow
    root        : middle ground — faster ramp than exponential
    """

    def __init__(
        self,
        n_samples:         int,
        rare_doc_mask:     np.ndarray,       # bool array, True if doc has rare labels
        initial_ratio:     float = SPL_INITIAL_RATIO,
        final_ratio:       float = SPL_FINAL_RATIO,
        min_ratio:         float = SPL_MIN_RATIO,
        pacing:            str   = SPL_PACING,
        rare_boost:        float = SPL_RARE_BOOST,
    ):
        self.n_samples     = n_samples
        self.rare_doc_mask = rare_doc_mask.astype(bool)
        self.initial_ratio = initial_ratio
        self.final_ratio   = final_ratio
        self.min_ratio     = min_ratio
        self.pacing        = pacing
        self.rare_boost    = rare_boost

        # Per-document loss estimates; start high (include nothing until first update)
        self.sample_losses = np.full(n_samples, float("inf"))
        self._losses_initialised = False

    # ----------------------------------------------------------
    # Pacing schedule
    # ----------------------------------------------------------
    def _pace(self, t: float) -> float:
        """Map normalised time t ∈ [0,1] → inclusion ratio."""
        t = float(np.clip(t, 0.0, 1.0))
        if self.pacing == "linear":
            ratio = self.initial_ratio + (self.final_ratio - self.initial_ratio) * t
        elif self.pacing == "exponential":
            # Geometric interpolation in log-space
            log_i = np.log(max(self.initial_ratio, 1e-6))
            log_f = np.log(max(self.final_ratio,   1e-6))
            ratio = float(np.exp(log_i + (log_f - log_i) * t))
        elif self.pacing == "logarithmic":
            # ln(1 + 9t) / ln(10) maps [0,1] → [0,1] with log shape
            ratio = self.initial_ratio + (self.final_ratio - self.initial_ratio) * (
                np.log1p(9.0 * t) / np.log(10.0)
            )
        elif self.pacing == "root":
            ratio = self.initial_ratio + (self.final_ratio - self.initial_ratio) * np.sqrt(t)
        else:
            raise ValueError(f"Unknown SPL_PACING: {self.pacing!r}")
        return float(np.clip(ratio, self.min_ratio, self.final_ratio))

    def current_ratio(self, epoch: int, total_epochs: int) -> float:
        t = (epoch - 1) / max(1, total_epochs - 1)
        return self._pace(t)

    # ----------------------------------------------------------
    # Loss update
    # ----------------------------------------------------------
    def update_losses(
        self,
        model:   nn.Module,
        dataset: Dataset,
        device:  str,
    ) -> None:
        """
        Recompute per-document loss for every sample in the dataset.
        Called at the start of each epoch (or every SPL_UPDATE_FREQ epochs).
        """
        model.eval()
        loader = DataLoader(
            dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc
        )
        new_losses = []
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                token_type_ids = token_type_ids.to(device)
                labels         = labels.to(device)
                lengths        = lengths.to(device)

                # Per-document loss via individual forward passes in this batch
                B = input_ids.size(0)
                for b in range(B):
                    ids_b   = input_ids[b:b+1]
                    mask_b  = attention_mask[b:b+1]
                    types_b = token_type_ids[b:b+1]
                    labs_b  = labels[b:b+1]
                    lens_b  = lengths[b:b+1]
                    loss_b, _ = model(
                        ids_b, mask_b, types_b,
                        labels=labs_b, lengths=lens_b,
                    )
                    val = loss_b.item() if not torch.isnan(loss_b) else 1e6
                    new_losses.append(val)

        self.sample_losses    = np.array(new_losses, dtype=np.float64)
        self._losses_initialised = True
        model.train()

    # ----------------------------------------------------------
    # Sample selection
    # ----------------------------------------------------------
    def select_indices(self, epoch: int, total_epochs: int) -> list:
        """
        Return the list of dataset indices to train on for this epoch.

        Rare-class documents get their loss divided by `rare_boost`,
        making them appear easier → they enter the curriculum sooner.
        """
        if not self._losses_initialised:
            # Before first loss update, return all indices
            return list(range(self.n_samples))

        ratio     = self.current_ratio(epoch, total_epochs)
        n_select  = max(int(self.min_ratio * self.n_samples),
                        int(ratio * self.n_samples))
        n_select  = min(n_select, self.n_samples)

        # Adjust losses: divide rare-class doc losses by boost factor
        adjusted = self.sample_losses.copy()
        adjusted[self.rare_doc_mask] /= self.rare_boost

        # Sort ascending (easiest first) and take top-k
        sorted_idx = np.argsort(adjusted)
        selected   = sorted_idx[:n_select].tolist()
        return selected

    # ----------------------------------------------------------
    # Diagnostics
    # ----------------------------------------------------------
    def diagnostics(self, epoch: int, total_epochs: int) -> dict:
        """Return a dict of SPL stats for this epoch (logged to history)."""
        ratio    = self.current_ratio(epoch, total_epochs)
        selected = self.select_indices(epoch, total_epochs)
        sel_set  = set(selected)
        rare_indices = np.where(self.rare_doc_mask)[0]
        rare_included = sum(1 for i in rare_indices if i in sel_set)
        rare_total    = int(rare_indices.sum()) if self.rare_doc_mask.any() else 0

        finite_losses = self.sample_losses[np.isfinite(self.sample_losses)]
        return {
            "spl_ratio":           round(ratio, 4),
            "spl_n_selected":      len(selected),
            "spl_n_total":         self.n_samples,
            "spl_rare_included":   rare_included,
            "spl_rare_total":      rare_total,
            "spl_rare_incl_rate":  round(rare_included / max(1, rare_total), 4),
            "spl_mean_loss":       round(float(finite_losses.mean()), 4)
                                   if len(finite_losses) > 0 else None,
            "spl_std_loss":        round(float(finite_losses.std()),  4)
                                   if len(finite_losses) > 0 else None,
        }


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":           "InLegalBERT Encoder",
        "sent_bilstm":    "Sentence BiLSTM",
        "mha_pooling":    "Multi-Head Attn Pooling",
        "ctx_bilstm":     "Context BiLSTM",
        "classifier":     "Classifier Head",
        "crf":            "CRF",
        "dropout":        "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })
    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v4 SPL)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(
            f"  {r['Component']:<30} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


def build_rare_doc_mask(docs: list, rare_ids: list) -> np.ndarray:
    """
    Return a boolean array of length len(docs).
    True if the document contains at least one sentence with a rare label.
    Used by SelfPacedLearner to boost rare-class documents.
    """
    rare_set = set(rare_ids)
    mask = np.array(
        [any(lab in rare_set for lab in labs) for _, labs in docs],
        dtype=bool,
    )
    n_rare_docs = mask.sum()
    print(f"   📌 SPL rare-doc mask: {n_rare_docs}/{len(docs)} docs "
          f"({n_rare_docs/max(1,len(docs))*100:.1f}%) contain ≥1 rare-class sentence.\n")
    return mask


def compute_class_weights(docs: list, rare_ids: list) -> torch.Tensor:
    """
    Inverse-frequency class weights for CE auxiliary loss.
    Rare classes receive higher weight.
    """
    all_labels = [lab for _, labs in docs for lab in labs]
    counts = Counter(all_labels)
    total  = sum(counts.values())
    weights = []
    for i in range(NUM_LABELS):
        freq = counts.get(i, 0) / max(1, total)
        # Smoothed inverse frequency; rare classes → large weight
        w = 1.0 / (freq + 1e-5)
        weights.append(w)
    # Normalise so mean weight == 1
    weights = torch.tensor(weights, dtype=torch.float)
    weights = weights / weights.mean()
    print("   📐 Class weights for CE loss:")
    for i, lbl in enumerate(LABELS):
        tag = " ← rare" if i in rare_ids else ""
        print(f"      {lbl:<20} w={weights[i].item():.3f}{tag}")
    print()
    return weights


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = None,  # NEW: optional Tensor[NUM_LABELS] for CE
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2     # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Auxiliary CE: class-weighted if SPL_USE_CLASS_WEIGHTS
        self.ce_loss = nn.CrossEntropyLoss(
            weight          = class_weights,   # NEW: weighted CE
            label_smoothing = LABEL_SMOOTHING,
            ignore_index    = -100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # ── After pack_padded_sequence → pad_packed_sequence, emissions has shape
        #    (B, actual_max_len, C) which may be SHORTER than labels (B, T_padded).
        #    We MUST trim labels to emissions' temporal dim before CRF / CE.
        T_emit = emissions.shape[1]

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :min(int(l.item()), T)] = True
        elif labels is not None:
            # Trim labels before building mask too
            mask = (labels[:, :T_emit] != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            # KEY FIX: trim labels to match emissions' temporal dimension
            labels_trimmed = labels[:, :T_emit]

            safe_labels = labels_trimmed.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels_trimmed.reshape(B2 * T2),
            )
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })
        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = m(input_ids, attention_mask, token_type_ids,
                            labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    # ----------------------------------------------------------
    # MAIN TRAINING LOOP  (SPL-aware)
    # ----------------------------------------------------------
    def train(
        self,
        train_dataset: Dataset,
        dev_dataset:   Dataset,
        rare_ids:      list,
        tokenizer,
        num_epochs:    int = NUM_EPOCHS,
        spl:           SelfPacedLearner = None,
    ):
        optimizer = self.build_optimizer()

        # Cosine annealing scheduler (v3)
        # We estimate total_steps from the full dataset; SPL reduces each epoch's
        # actual steps but the scheduler is based on the ceiling.
        total_steps  = (len(train_dataset) // BATCH_DOCS) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        # SWA (v3)
        swa_model    = AveragedModel(self.model)
        swa_start_ep = max(1, int(num_epochs * SWA_START_FRAC))
        swa_sched    = SWALR(optimizer, swa_lr=SWA_LR,
                             anneal_epochs=5, anneal_strategy="cos")
        swa_active   = False
        print(f"📊 SWA starts at epoch {swa_start_ep}/{num_epochs}.")

        # SPL state
        use_spl = SPL_ENABLED and spl is not None
        if use_spl:
            print(f"🎓 SPL enabled | pacing={SPL_PACING} | "
                  f"ratio {SPL_INITIAL_RATIO:.0%}→{SPL_FINAL_RATIO:.0%} | "
                  f"rare_boost={SPL_RARE_BOOST}x | "
                  f"update_freq={SPL_UPDATE_FREQ} epochs\n")

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None

        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch

            # ── SPL: recompute losses and select subset ────────
            spl_diag = {}
            if use_spl:
                # Recompute per-doc losses periodically
                if epoch == 1 or (epoch - 1) % SPL_UPDATE_FREQ == 0:
                    print(f"   [SPL] Recomputing per-doc losses (epoch {epoch})...")
                    spl.update_losses(self.model, train_dataset, self.device)

                selected_indices = spl.select_indices(epoch, num_epochs)
                active_dataset   = Subset(train_dataset, selected_indices)
                spl_diag         = spl.diagnostics(epoch, num_epochs)
                print(
                    f"   [SPL] epoch={epoch} | "
                    f"ratio={spl_diag['spl_ratio']:.2%} | "
                    f"docs={spl_diag['spl_n_selected']}/{spl_diag['spl_n_total']} | "
                    f"rare_included={spl_diag['spl_rare_included']}/{spl_diag['spl_rare_total']} "
                    f"({spl_diag['spl_rare_incl_rate']:.0%})"
                )
            else:
                active_dataset = train_dataset

            train_loader = DataLoader(
                active_dataset,
                batch_size  = BATCH_DOCS,
                shuffle     = True,
                collate_fn  = collate_rrc,
            )

            # ── Epoch training ─────────────────────────────────
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            # Final partial flush
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active:
                    scheduler.step()
                optimizer.zero_grad()

            # ── SWA update ─────────────────────────────────────
            if epoch >= swa_start_ep:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_sched.step()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan={nan_steps}]" if nan_steps > 0 else ""
            swa_tag  = " [SWA]" if swa_active else ""
            spl_tag  = f" [SPL {spl_diag.get('spl_ratio', 1.0):.0%}]" if use_spl else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train: {avg_train_loss:.4f} | "
                f"val: {val_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"acc: {val_metrics['accuracy']:.4f} | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{spl_tag}{swa_tag}{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "swa_active":              swa_active,
                "timestamp":               datetime.utcnow().isoformat(),
                # SPL diagnostics
                **spl_diag,
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        # ── Finalize SWA ───────────────────────────────────────
        if swa_active:
            print("📐 Updating SWA BatchNorm stats...")
            update_bn(
                DataLoader(train_dataset, batch_size=BATCH_DOCS,
                           shuffle=False, collate_fn=collate_rrc),
                swa_model,
                device=self.device,
            )
            swa_val_loss    = self.compute_val_loss(dev_dataset, model_override=swa_model)
            swa_val_metrics = self.evaluate(dev_dataset, rare_ids,
                                            model_override=swa_model)
            print(f"  SWA val_loss: {swa_val_loss:.4f} | "
                  f"SWA macro_f1: {swa_val_metrics['macro_f1']:.4f}")
            if swa_val_metrics["macro_f1"] > best_f1:
                best_f1    = swa_val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print(f"  ✔ SWA weights beat individual checkpoint — using SWA.")

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time: {total_train_time/60:.2f} min — "
              f"{actual_epochs} epochs")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "swa_start_epoch":         swa_start_ep,
            "spl_enabled":             use_spl,
            "spl_pacing":              SPL_PACING if use_spl else None,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                decoded, _ = m(input_ids, attention_mask, token_type_ids,
                               labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {total_infer_time:.2f}s | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
            "spl_enabled":      SPL_ENABLED,
            "spl_pacing":       SPL_PACING,
            "spl_rare_boost":   SPL_RARE_BOOST,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        swa_ep = None
        if "swa_active" in hist_df.columns:
            swa_rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not swa_rows.empty:
                swa_ep = int(swa_rows.iloc[0])

        # ── 1. Loss curves ─────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss",
                marker="o", markersize=3)
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",
                marker="s", markersize=3)
        if swa_ep:
            ax.axvline(swa_ep, color="green", linestyle="--", alpha=0.6,
                       label=f"SWA starts (ep {swa_ep})")
        ax.set_title("Training vs Validation Loss (v4 SPL)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        # ── 2. F1 curves ───────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, markersize=3)
        if swa_ep:
            axes[0].axvline(swa_ep, color="green", linestyle="--", alpha=0.5)
        axes[0].set_title("Validation F1 Scores")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss",
                     marker="o", markersize=3)
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",
                     marker="s", markersize=3)
        if swa_ep:
            axes[1].axvline(swa_ep, color="green", linestyle="--", alpha=0.5)
        axes[1].set_title("Loss Curves")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        # ── 3. Precision / Recall ──────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[0].set_title("Validation Precision")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[1].set_title("Validation Recall")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        # ── 4. SPL diagnostics (NEW) ───────────────────────────
        if "spl_ratio" in hist_df.columns and hist_df["spl_ratio"].notna().any():
            fig, axes = plt.subplots(1, 2, figsize=(14, 4))

            axes[0].plot(epochs, hist_df["spl_ratio"], color="darkorange",
                         marker="o", markersize=3, label="SPL ratio")
            axes[0].set_ylim(0, 1.05)
            axes[0].set_title(f"SPL Pacing Curve ({SPL_PACING})")
            axes[0].set_xlabel("Epoch")
            axes[0].set_ylabel("Fraction of training data used")
            axes[0].legend(); axes[0].grid(True, alpha=0.3)

            if "spl_rare_incl_rate" in hist_df.columns:
                axes[1].plot(epochs, hist_df["spl_rare_incl_rate"],
                             color="tomato", marker="s", markersize=3,
                             label="Rare-class doc inclusion rate")
                axes[1].plot(epochs, hist_df["spl_ratio"],
                             color="darkorange", linestyle="--", markersize=3,
                             label="Overall SPL ratio")
                axes[1].set_ylim(0, 1.05)
                axes[1].set_title("Rare-Class Document Inclusion (SPL Boost)")
                axes[1].set_xlabel("Epoch")
                axes[1].set_ylabel("Inclusion rate")
                axes[1].legend(); axes[1].grid(True, alpha=0.3)

            plt.tight_layout()
            p = os.path.join(OUT_DIR, "spl_diagnostics.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        # ── 5. Epoch time ──────────────────────────────────────
        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"],
                   color="steelblue", alpha=0.8)
            ax.axhline(hist_df["epoch_train_time_s"].mean(), color="red",
                       linestyle="--",
                       label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s")
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA-Pool + CRF v4 SPL)")
    print("=" * 68)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print(f"  SPL Pacing           : {SPL_PACING} "
          f"({SPL_INITIAL_RATIO:.0%} → {SPL_FINAL_RATIO:.0%}, "
          f"rare_boost={SPL_RARE_BOOST}x)")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        sep = "─" * 68 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT → Sentence BiLSTM(128,L=1) → "
          "MHA(4-head) → Context BiLSTM(64,L=1) → Linear → CRF + Aux-CE")
    print(f"\nv4 additions on top of v3:")
    print(f"  SPL_ENABLED           : {SPL_ENABLED}")
    print(f"  SPL_PACING            : {SPL_PACING}")
    print(f"  SPL_INITIAL_RATIO     : {SPL_INITIAL_RATIO}")
    print(f"  SPL_FINAL_RATIO       : {SPL_FINAL_RATIO}")
    print(f"  SPL_RARE_BOOST        : {SPL_RARE_BOOST}x")
    print(f"  SPL_MIN_RATIO         : {SPL_MIN_RATIO}")
    print(f"  SPL_UPDATE_FREQ       : every {SPL_UPDATE_FREQ} epoch(s)")
    print(f"  SPL_USE_CLASS_WEIGHTS : {SPL_USE_CLASS_WEIGHTS}\n")

    # ── Load data ──────────────────────────────────────────
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    # ── Class analysis ─────────────────────────────────────
    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── SPL: build rare-doc mask ───────────────────────────
    rare_doc_mask = build_rare_doc_mask(train_docs, rare_ids)

    # ── Class weights for CE loss ──────────────────────────
    class_weights = None
    if SPL_USE_CLASS_WEIGHTS:
        class_weights = compute_class_weights(train_docs, rare_ids).to(DEVICE)

    # ── SPL learner ────────────────────────────────────────
    spl = SelfPacedLearner(
        n_samples     = len(train_docs),
        rare_doc_mask = rare_doc_mask,
        initial_ratio = SPL_INITIAL_RATIO,
        final_ratio   = SPL_FINAL_RATIO,
        min_ratio     = SPL_MIN_RATIO,
        pacing        = SPL_PACING,
        rare_boost    = SPL_RARE_BOOST,
    ) if SPL_ENABLED else None

    # ── Tokeniser & datasets ───────────────────────────────
    print("Loading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── Model ──────────────────────────────────────────────
    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = class_weights,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    # ── Training ───────────────────────────────────────────
    trainer = Trainer(model, device=DEVICE)
    print(f"\nStarting training (max {NUM_EPOCHS} epochs, ES patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset,
        dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
        spl        = spl,
    )
    print("\nTraining complete.")

    # ── Load best checkpoint ───────────────────────────────
    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    # ── Dev evaluation ─────────────────────────────────────
    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v4 (SPL)\n")
        f.write(f"SPL: pacing={SPL_PACING}, rare_boost={SPL_RARE_BOOST}x\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    # ── Test evaluation ────────────────────────────────────
    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v4 (SPL)\n")
        f.write(f"SPL: pacing={SPL_PACING}, rare_boost={SPL_RARE_BOOST}x\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        pd.DataFrame([
            {
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        mets["per_class_metrics"][lbl]["f1"],
                "precision": mets["per_class_metrics"][lbl]["precision"],
                "recall":    mets["per_class_metrics"][lbl]["recall"],
            }
            for lbl in LABELS
        ]).to_csv(os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA-Pooling + CRF v4 SPL",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
        },
        "spl": {
            "enabled":      SPL_ENABLED,
            "pacing":       SPL_PACING,
            "initial_ratio": SPL_INITIAL_RATIO,
            "final_ratio":  SPL_FINAL_RATIO,
            "rare_boost":   SPL_RARE_BOOST,
            "update_freq":  SPL_UPDATE_FREQ,
            "class_weights_used": SPL_USE_CLASS_WEIGHTS,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT → Sentence BiLSTM(128,L=1) → MHA(4-head) → Context BiLSTM(64,L=1) → Linear → CRF + Aux-CE

v4 additions on top of v3:
  SPL_ENABLED           : True
  SPL_PACING            : exponential
  SPL_INITIAL_RATIO     : 0.4
  SPL_FINAL_RATIO       : 1.0
  SPL_RARE_BOOST        : 2.0x
  SPL_MIN_RATIO         : 0.15
  SPL_UPDATE_FREQ       : every 2 epoch(s)
  SPL_USE_CLASS_WEIGHTS : True

  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-9.
🔥 BERT layers trainable: layers 10-11 (2 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v4 SPL)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder                14,766,336 94,715,904  109,482,240
  Sentence BiLSTM                       919,552          0      919,552
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        164,864          0      164,864
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
  Dropout                                     0          0            0
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        16,057,680 94,715,904  110,773,584

Starting training (max 80 

/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1124: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1136: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(


  SWA val_loss: 72.5282 | SWA macro_f1: 0.4028

⏱  Total training time: 54.06 min — 80 epochs
Saved rrc_bilstm_mha_crf_spl_v4_logs/training_loss_curve.png
Saved rrc_bilstm_mha_crf_spl_v4_logs/training_f1_curve.png
Saved rrc_bilstm_mha_crf_spl_v4_logs/training_pr_curve.png
Saved rrc_bilstm_mha_crf_spl_v4_logs/spl_diagnostics.png
Saved rrc_bilstm_mha_crf_spl_v4_logs/training_time_curve.png

💾 Best model saved → rrc_bilstm_mha_crf_spl_v4_logs/best_model/

Training complete.
Loaded best checkpoint.

Evaluating on Dev set...

⏱  Inference (dev): 1.60s | throughput: 1804.3 sent/s
  Dev  Accuracy : 0.7761
  Dev  Macro-F1 : 0.4339
  Dev  Rare-F1  : 0.3025
Saved rrc_bilstm_mha_crf_spl_v4_logs/dev_confusion_matrix.png
Saved rrc_bilstm_mha_crf_spl_v4_logs/dev_per_class_f1.png

Evaluating on Test set...

⏱  Inference (test): 2.33s | throughput: 1788.3 sent/s
  Test Accuracy : 0.8211
  Test Macro-F1 : 0.4742
  Test Rare-F1  : 0.3456
Saved rrc_bilstm_mha_crf_spl_v4_logs/test_confusion_matrix.png
Sav